In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity
from exponential_smoothing import build_exponential_smoothing_correlation_graph

import networkx as nx
import matplotlib.pyplot as plt



In [2]:
DATA_PATH = '../dataset/data_andre.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df['item_id'] = df['item_id'].astype(int) # or .astype(str)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]
print(len(y))

# Calandar-based features
# Ensure datetime
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

# Base calendar parts
df["day_of_week"]  = df[DATE_COL].dt.dayofweek.astype(int)
df["day_of_month"] = df[DATE_COL].dt.day.astype(int)         
df["moy"]          = df[DATE_COL].dt.month.astype(int)-1      
df["doy"]          = df[DATE_COL].dt.dayofyear.astype(int)-1   
df["is_weekend"] = (
    (df[DATE_COL].dt.dayofweek == 5) |
    (df[DATE_COL].dt.dayofweek == 6)
).astype(int)

df = df.sort_values(["item_id", DATE_COL]).reset_index(drop=True)

df

Loading data from ../dataset/data_andre.feather...
1082371


,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,promo_type_CLCP,promo_value_CLCP,promo_type_LFPE,promo_value_LFPE,store_id,day_of_week,day_of_month,moy,doy,is_weekend
0,2021-01-23,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0.0,0,0.0,6269,5,23,0,22,1
1,2021-01-24,27,14,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0.0,0,0.0,6269,6,24,0,23,1
2,2021-01-25,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0.0,0,0.0,6269,0,25,0,24,0
3,2021-01-26,27,5,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0.0,0,0.0,6269,1,26,0,25,0
4,2021-01-27,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0.0,0,0.0,6269,2,27,0,26,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1082366,2023-02-18,900087600,28,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0.0,0,0.0,6269,5,18,1,48,1
1082367,2023-02-19,900087600,46,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0.0,0,0.0,6269,6,19,1,49,1
1082368,2023-02-20,900087600,28,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0.0,0,0.0,6269,0,20,1,50,0
1082369,2023-02-21,900087600,31,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0.0,0,0.0,6269,1,21,1,51,0


In [3]:
G, corr_mat, residuals_df = build_exponential_smoothing_correlation_graph(
    df, 
    seasonal_periods=7,
    trend="add",
    seasonal="add",
    corr_threshold=0.8
)
print(f"Graph nodes: {G.number_of_nodes()}, Graph edges: {G.number_of_edges()}")

Graph nodes: 1427, Graph edges: 0


In [4]:
for edge in G.edges(data=True):
    #print ids of the edge
    df_edge = df[df['item_id'].isin([edge[0], edge[1]])]
    print(df_edge)
    
    